# Lab 04-07: MCP Servers

**Module:** 04 — Assembling and Deploying Applications  
**Exam Weight:** ~10 questions across Module 04 (22% of exam)  
**UI Sections:** Playground | Catalog | Serving  
**Difficulty:** Intermediate  
**Estimated Time:** 40–50 minutes

---

## Learning Objectives

- Understand **Model Context Protocol (MCP)** and how it gives agents access to external tools
- Distinguish the **three MCP server types**: Managed, External, and Custom
- Know which Databricks features are exposed as **Managed MCP servers** (Unity Catalog, Genie)
- Understand how agents **discover and invoke** tools via MCP at runtime
- Build a decision framework for choosing the right MCP server type

---

## Exam Objectives Covered

- **Identify the three types of MCP servers and when to use each**
- **Describe how agents discover and use tools through MCP**
- **Configure an agent to use MCP-provided tools**

---

## What Is MCP and Why Does It Matter?

The **Model Context Protocol (MCP)** is an open standard that defines how AI agents connect to external tools and data sources. Think of it as a universal "plug" that lets any agent talk to any tool, without custom integration code for each combination.

Without MCP, every agent-tool connection requires custom code:

```
BEFORE MCP (custom integration for each tool):

Agent A --[custom code]--> Tool 1 (SQL)
Agent A --[custom code]--> Tool 2 (Slack)
Agent B --[custom code]--> Tool 1 (SQL)   <-- duplicate work!
Agent B --[custom code]--> Tool 3 (GitHub)
```

With MCP, tools expose a standard interface, and any agent can use any tool:

```
AFTER MCP (standardized protocol):

Agent A --[MCP]--> MCP Server 1 (SQL)
Agent A --[MCP]--> MCP Server 2 (Slack)
Agent B --[MCP]--> MCP Server 1 (SQL)   <-- same interface, no duplicate code
Agent B --[MCP]--> MCP Server 3 (GitHub)
```

### MCP Architecture: Server vs Client

| Component | Role | Example |
|-----------|------|--------|
| **MCP Server** | Exposes tools and data to agents | Unity Catalog MCP, Slack MCP, custom API MCP |
| **MCP Client** | The agent that discovers and calls tools | Your LangChain agent, Databricks Agent Framework |

The agent (client) asks the MCP server: "What tools do you have?" The server responds with a list. The agent picks the right tool and calls it through the MCP protocol.

---

## Mental Model

> **Analogy:** MCP servers are like USB adapters for AI agents. **Managed MCP** = built-in USB ports that come with your laptop (Databricks provides them — Unity Catalog, Genie). **External MCP** = off-the-shelf adapters you buy at the store (pre-built connectors for Slack, GitHub, etc.). **Custom MCP** = you solder your own adapter for a unique device (your company's proprietary API or internal system).

All three adapter types plug into the same USB standard (MCP protocol). The agent does not need to know which type of adapter it is using — it just discovers tools and calls them.

---

## Exam Alert: Common Traps

| # | Trap Statement | Correct Answer |
|---|---------------|----------------|
| 1 | "MCP is only for custom integrations" | **Wrong.** Databricks provides managed MCP servers for Unity Catalog and Genie out-of-the-box. No custom code needed. |
| 2 | "Agents can only use tools defined in their code" | **Wrong.** MCP servers provide tools dynamically — agents discover them at runtime. An agent can use a new tool without code changes. |
| 3 | "All MCP servers are hosted by Databricks" | **Wrong.** Only managed MCP servers are hosted by Databricks. External MCP servers connect to third-party services. Custom MCP servers run wherever you deploy them. |
| 4 | "MCP replaces Unity Catalog functions" | **Wrong.** Managed MCP *exposes* UC functions as tools. UC functions are still the underlying implementation. |
| 5 | "You need to write MCP protocol code from scratch" | **Wrong.** Use the MCP SDK — it handles the protocol. You just define your tool functions. |

---

## Prerequisites and UI Navigation

### What You Need
- A Databricks workspace with Unity Catalog enabled
- Access to a cluster with Databricks Runtime ML
- Familiarity with Unity Catalog functions (from Module 03)

### Key UI Paths
- **UI -->** Left nav --> **Catalog** --> Browse functions that become MCP tools
- **UI -->** Left nav --> **Playground** --> Test agents with MCP tools interactively
- **UI -->** Left nav --> **Serving** --> View serving endpoints with MCP configuration

---

## Setup

In [ ]:
# Install required packages
%pip install databricks-sdk -q
dbutils.library.restartPython()

In [ ]:
# ---- Imports and Configuration ----
import os
import json
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Configuration
CATALOG = "workspace"
SCHEMA = "genai_labs"
MODEL_CHAT = "databricks-meta-llama-3-3-70b-instruct"

# Ensure schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

print(f"Catalog: {CATALOG}")
print(f"Schema:  {SCHEMA}")
print("Setup complete.")

### Expected Output

```
Catalog: main
Schema:  genai_labs
Setup complete.
```

### What Just Happened

We set up the workspace and ensured our Unity Catalog schema exists. This schema will hold the UC functions that become available as MCP tools.

---

## Step 1: What Is MCP? — Protocol Overview

MCP defines a standard way for agents to:
1. **Discover** available tools ("What can I do?")
2. **Understand** tool parameters ("What inputs does this tool need?")
3. **Invoke** tools ("Run this tool with these inputs")
4. **Receive** results ("Here is the tool's output")

### The MCP Request/Response Flow

```
Agent (MCP Client)                    MCP Server
    |                                      |
    |--- 1. list_tools() ----------------->|
    |<-- 2. [tool_a, tool_b, tool_c] ------|
    |                                      |
    |--- 3. call_tool(tool_a, params) ---->|
    |<-- 4. {result: "..."} ---------------|
    |                                      |
```

### The Three Types of MCP Servers on Databricks

| Type | Description | Hosted By | Examples |
|------|-------------|-----------|----------|
| **Managed** | Built-in, provided by Databricks | Databricks | Unity Catalog MCP, Genie MCP |
| **External** | Pre-built connectors for third-party services | Third party / Databricks | Slack, GitHub, Jira, Google Drive |
| **Custom** | You build and host the MCP server | You | Proprietary APIs, internal databases, legacy systems |

In [ ]:
# ---- Step 1: The three MCP server types explained ----

mcp_types = [
    {
        "type": "Managed MCP",
        "description": "Built-in MCP servers provided by Databricks",
        "hosted_by": "Databricks (zero setup)",
        "examples": ["Unity Catalog MCP (table/function access)", "Genie MCP (NL-to-SQL)"],
        "when_to_use": "Access Databricks-native resources (tables, functions, SQL)",
        "setup_effort": "None -- automatically available"
    },
    {
        "type": "External MCP",
        "description": "Pre-built connectors for third-party services",
        "hosted_by": "Third party or Databricks marketplace",
        "examples": ["Slack MCP", "GitHub MCP", "Jira MCP", "Google Drive MCP"],
        "when_to_use": "Integrate with common SaaS tools your team already uses",
        "setup_effort": "Low -- configure credentials and permissions"
    },
    {
        "type": "Custom MCP",
        "description": "You build and host an MCP server for proprietary systems",
        "hosted_by": "Your infrastructure (or Databricks Apps)",
        "examples": ["Internal CRM API", "Proprietary data warehouse", "Custom ML models"],
        "when_to_use": "No pre-built connector exists for your system",
        "setup_effort": "Medium-High -- write server code, deploy, maintain"
    },
]

print("=== The Three Types of MCP Servers ===")
print()
for mcp in mcp_types:
    print(f"  {mcp['type']}")
    print(f"    Description:  {mcp['description']}")
    print(f"    Hosted by:    {mcp['hosted_by']}")
    print(f"    Examples:     {', '.join(mcp['examples'])}")
    print(f"    When to use:  {mcp['when_to_use']}")
    print(f"    Setup effort: {mcp['setup_effort']}")
    print()

### Expected Output

```
=== The Three Types of MCP Servers ===

  Managed MCP
    Description:  Built-in MCP servers provided by Databricks
    Hosted by:    Databricks (zero setup)
    Examples:     Unity Catalog MCP (table/function access), Genie MCP (NL-to-SQL)
    When to use:  Access Databricks-native resources (tables, functions, SQL)
    Setup effort: None -- automatically available

  External MCP
    Description:  Pre-built connectors for third-party services
    ...

  Custom MCP
    Description:  You build and host an MCP server for proprietary systems
    ...
```

### What Just Happened

We laid out the **three MCP server types** with their key characteristics. This is the foundational knowledge for the exam — you need to identify which type to use for a given scenario.

---

## Step 2: Managed MCP Servers — Unity Catalog and Genie

Managed MCP servers are **built into Databricks** and require zero setup. They automatically expose Databricks-native resources as tools that agents can discover and use.

### Unity Catalog MCP

The Unity Catalog MCP server exposes **UC functions** as agent tools. When you create a SQL or Python function in Unity Catalog, it automatically becomes available as an MCP tool. The agent can discover it, understand its parameters, and call it.

```
You create a UC function:
    CREATE FUNCTION workspace.genai_labs.get_order_status(order_id STRING)
    RETURNS STRING
    ...

MCP automatically exposes it:
    Tool: workspace.genai_labs.get_order_status
    Description: Returns the status of an order
    Parameters: {order_id: STRING}

Agent discovers and calls it:
    Agent: "I need to check order ORD-123"
    --> calls get_order_status("ORD-123")
    --> returns "Shipped - tracking: TRK-456"
```

### Genie MCP

The Genie MCP server gives agents **natural language to SQL** capabilities. The agent can ask a question in plain English, and Genie translates it to SQL, runs it against your data, and returns the result.

| Managed MCP Server | What It Exposes | Use Case |
|-------------------|-----------------|----------|
| **Unity Catalog MCP** | UC functions as callable tools | Agent needs to run specific, well-defined operations |
| **Genie MCP** | Natural language SQL queries | Agent needs to answer ad-hoc data questions |

In [ ]:
# ---- Step 2a: Create a UC function that becomes an MCP tool ----
# When you create a UC function, it automatically becomes available
# as a tool through the Unity Catalog MCP server.

# Create a simple lookup function
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_order_status(order_id STRING)
RETURNS STRING
LANGUAGE SQL
COMMENT 'Returns the current status of a customer order. Use when a customer asks about their order.'
RETURN CASE
    WHEN order_id = 'ORD-001' THEN 'Shipped - Tracking: TRK-789'
    WHEN order_id = 'ORD-002' THEN 'Processing - Estimated ship: 3 days'
    WHEN order_id = 'ORD-003' THEN 'Delivered - Signed by: J. Smith'
    ELSE 'Order not found. Please verify the order ID.'
END
""")

# Create a calculator function
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.calculate_discount(
    original_price DOUBLE,
    discount_percent DOUBLE
)
RETURNS DOUBLE
LANGUAGE SQL
COMMENT 'Calculates the discounted price. Use when a customer asks about pricing with a discount.'
RETURN original_price * (1 - discount_percent / 100)
""")

# Test the functions
print("=== UC Functions Created (Available as MCP Tools) ===")
print()

result1 = spark.sql(f"SELECT {CATALOG}.{SCHEMA}.get_order_status('ORD-001') AS status").collect()[0][0]
print(f"  get_order_status('ORD-001') = {result1}")

result2 = spark.sql(f"SELECT {CATALOG}.{SCHEMA}.calculate_discount(100.0, 15.0) AS price").collect()[0][0]
print(f"  calculate_discount(100.0, 15.0) = ${result2:.2f}")

print()
print("These functions are now automatically discoverable as MCP tools.")
print("Navigate to: **UI -->** Catalog --> main --> genai_labs --> Functions")

### Expected Output

```
=== UC Functions Created (Available as MCP Tools) ===

  get_order_status('ORD-001') = Shipped - Tracking: TRK-789
  calculate_discount(100.0, 15.0) = $85.00

These functions are now automatically discoverable as MCP tools.
Navigate to: **UI -->** Catalog --> main --> genai_labs --> Functions
```

### What Just Happened

We created two **Unity Catalog functions** that are automatically exposed as MCP tools through the managed Unity Catalog MCP server. Key details:
- The `COMMENT` field becomes the tool's **description** — this is what the LLM reads to decide when to use the tool
- The function signature (parameters and return type) becomes the tool's **schema**
- No additional MCP configuration needed — UC functions are MCP tools by default

In [ ]:
# ---- Step 2b: List UC functions as MCP-discoverable tools ----
# An agent discovers tools by listing available UC functions.
# This simulates what happens when an agent connects to the UC MCP server.

functions_df = spark.sql(f"""
    SHOW FUNCTIONS IN {CATALOG}.{SCHEMA}
""")

print("=== MCP Tool Discovery: UC Functions ===")
print()
print("When an agent connects to the Unity Catalog MCP server,")
print("it discovers these tools:")
print()

functions = functions_df.collect()
for func in functions:
    func_name = func[0]
    # Get function details
    try:
        details = spark.sql(f"DESCRIBE FUNCTION EXTENDED {func_name}").collect()
        desc = next((row[0] for row in details if 'Comment:' in str(row[0])), "No description")
        print(f"  Tool: {func_name}")
        print(f"  {desc}")
        print()
    except Exception:
        print(f"  Tool: {func_name}")
        print()

### Expected Output

```
=== MCP Tool Discovery: UC Functions ===

When an agent connects to the Unity Catalog MCP server,
it discovers these tools:

  Tool: workspace.genai_labs.get_order_status
  Comment: Returns the current status of a customer order...

  Tool: workspace.genai_labs.calculate_discount
  Comment: Calculates the discounted price...
```

### What Just Happened

We simulated **MCP tool discovery** — the process by which an agent learns what tools are available. In production, the agent connects to the UC MCP server, receives the list of functions with their descriptions and schemas, and uses this information to decide which tool to call for each user request.

---

## Step 3: External MCP Servers

External MCP servers are **pre-built connectors** that integrate third-party services into your agent's toolkit. Databricks provides a growing catalog of external MCP servers.

### Available External MCP Servers

| Service | MCP Server | Tools Provided |
|---------|-----------|----------------|
| **Slack** | Slack MCP | Send messages, read channels, search conversations |
| **GitHub** | GitHub MCP | Create issues, read PRs, search repositories |
| **Jira** | Jira MCP | Create tickets, update status, search issues |
| **Google Drive** | Google Drive MCP | Search files, read documents, list folders |
| **Confluence** | Confluence MCP | Search pages, read content, list spaces |

### How to Connect an External MCP Server

```
1. Choose the external MCP server from the catalog
2. Configure authentication (OAuth, API key, etc.)
3. Set permissions (which tools the agent can use)
4. Connect the MCP server to your agent's configuration
5. The agent discovers new tools at runtime
```

In [ ]:
# ---- Step 3: External MCP server configuration (conceptual) ----
# External MCP servers connect to third-party services.
# This shows the configuration pattern.

# Example: How an external MCP server (Slack) would be configured
external_mcp_config = {
    "server_type": "external",
    "name": "slack-mcp-server",
    "provider": "Slack",
    "authentication": {
        "method": "oauth2",
        "credentials": "{{secrets/my-scope/slack-oauth-token}}"
    },
    "tools_exposed": [
        {
            "name": "send_message",
            "description": "Send a message to a Slack channel",
            "parameters": {"channel": "string", "message": "string"}
        },
        {
            "name": "search_messages",
            "description": "Search for messages across Slack channels",
            "parameters": {"query": "string", "limit": "integer"}
        },
        {
            "name": "list_channels",
            "description": "List available Slack channels",
            "parameters": {}
        },
    ]
}

print("=== External MCP Server Configuration Example (Slack) ===")
print()
print(f"  Server: {external_mcp_config['name']}")
print(f"  Provider: {external_mcp_config['provider']}")
print(f"  Auth: {external_mcp_config['authentication']['method']}")
print(f"  Credentials: {external_mcp_config['authentication']['credentials']}")
print()
print("  Tools exposed to agents:")
for tool in external_mcp_config['tools_exposed']:
    params = ', '.join(f"{k}: {v}" for k, v in tool['parameters'].items())
    print(f"    - {tool['name']}({params})")
    print(f"      {tool['description']}")
print()
print("KEY POINT: API keys are stored in Databricks Secrets, NOT in code.")

### Expected Output

```
=== External MCP Server Configuration Example (Slack) ===

  Server: slack-mcp-server
  Provider: Slack
  Auth: oauth2
  Credentials: {{secrets/my-scope/slack-oauth-token}}

  Tools exposed to agents:
    - send_message(channel: string, message: string)
      Send a message to a Slack channel
    - search_messages(query: string, limit: integer)
      Search for messages across Slack channels
    - list_channels()
      List available Slack channels

KEY POINT: API keys are stored in Databricks Secrets, NOT in code.
```

### What Just Happened

We showed the **configuration pattern** for an external MCP server. The key points are:
- Authentication uses **Databricks Secrets** (never hardcode credentials)
- The MCP server exposes a set of **tools** with descriptions and parameter schemas
- The agent discovers these tools at runtime and decides which to call

---

## Step 4: Custom MCP Servers — Building Your Own

When no managed or external MCP server exists for your system, you build a **custom MCP server**. This is common for:
- Proprietary internal APIs
- Legacy databases
- Custom ML models
- Industry-specific systems (EHR, ERP, etc.)

### Custom MCP Server Structure

```
Custom MCP Server
    |
    +-- Tool definitions (what tools are available)
    +-- Tool implementations (what each tool does)
    +-- MCP protocol handler (handles discovery and invocation)
    +-- Authentication (validates agent credentials)
```

In [ ]:
# ---- Step 4a: Custom MCP server example (Python) ----
# This shows how you would build a custom MCP server.
# In production, you would deploy this as a Databricks App or external service.

# The MCP SDK provides the framework -- you just define your tools.
# pip install mcp  (MCP Python SDK)

# Here is what a custom MCP server looks like conceptually:

custom_mcp_server_code = '''
from mcp.server import Server
from mcp.types import Tool, TextContent

# Create the MCP server
server = Server("internal-crm-mcp")

@server.list_tools()
async def list_tools():
    """Return the list of tools this server provides."""
    return [
        Tool(
            name="lookup_customer",
            description="Look up customer information by ID or email.",
            inputSchema={
                "type": "object",
                "properties": {
                    "customer_id": {"type": "string", "description": "Customer ID"},
                    "email": {"type": "string", "description": "Customer email"},
                },
                "required": ["customer_id"]
            }
        ),
        Tool(
            name="create_ticket",
            description="Create a support ticket in the internal CRM.",
            inputSchema={
                "type": "object",
                "properties": {
                    "customer_id": {"type": "string"},
                    "subject": {"type": "string"},
                    "priority": {"type": "string", "enum": ["low", "medium", "high"]},
                },
                "required": ["customer_id", "subject"]
            }
        ),
    ]

@server.call_tool()
async def call_tool(name, arguments):
    """Handle tool invocation."""
    if name == "lookup_customer":
        # Call your internal CRM API
        customer = crm_api.get_customer(arguments["customer_id"])
        return [TextContent(text=json.dumps(customer))]
    elif name == "create_ticket":
        ticket = crm_api.create_ticket(**arguments)
        return [TextContent(text=f"Ticket created: {ticket.id}")]
'''

print("=== Custom MCP Server Code Pattern ===")
print(custom_mcp_server_code)
print()
print("KEY POINTS:")
print("  1. Use the MCP SDK (pip install mcp) -- handles protocol details")
print("  2. Define tools with @server.list_tools() -- returns tool schemas")
print("  3. Implement tools with @server.call_tool() -- executes tool logic")
print("  4. Deploy as a Databricks App or external service")

### Expected Output

```
=== Custom MCP Server Code Pattern ===

from mcp.server import Server
from mcp.types import Tool, TextContent

# Create the MCP server
server = Server("internal-crm-mcp")

@server.list_tools()
async def list_tools():
    ...

@server.call_tool()
async def call_tool(name, arguments):
    ...

KEY POINTS:
  1. Use the MCP SDK (pip install mcp) -- handles protocol details
  2. Define tools with @server.list_tools() -- returns tool schemas
  3. Implement tools with @server.call_tool() -- executes tool logic
  4. Deploy as a Databricks App or external service
```

### What Just Happened

We showed the **code pattern** for building a custom MCP server. The MCP SDK handles the protocol — you just define two things:
1. `list_tools()` — What tools does your server offer?
2. `call_tool()` — What happens when an agent calls each tool?

In [ ]:
# ---- Step 4b: Simulate a custom MCP server locally ----
# We simulate what a custom MCP server does without the actual MCP SDK.

class SimulatedCRMMCPServer:
    """Simulates a custom MCP server for an internal CRM system."""
    
    def __init__(self):
        self.name = "internal-crm-mcp"
        # Simulated CRM database
        self.customers = {
            "C001": {"name": "Alice Johnson", "tier": "Gold", "email": "alice@example.com"},
            "C002": {"name": "Bob Smith", "tier": "Silver", "email": "bob@example.com"},
            "C003": {"name": "Carol Lee", "tier": "Platinum", "email": "carol@example.com"},
        }
        self.ticket_counter = 1000
    
    def list_tools(self):
        """MCP: Return available tools."""
        return [
            {"name": "lookup_customer", "description": "Look up customer by ID",
             "parameters": {"customer_id": "string"}},
            {"name": "create_ticket", "description": "Create a support ticket",
             "parameters": {"customer_id": "string", "subject": "string", "priority": "string"}},
        ]
    
    def call_tool(self, name, arguments):
        """MCP: Execute a tool."""
        if name == "lookup_customer":
            cid = arguments["customer_id"]
            return self.customers.get(cid, {"error": f"Customer {cid} not found"})
        elif name == "create_ticket":
            self.ticket_counter += 1
            return {"ticket_id": f"TKT-{self.ticket_counter}",
                    "customer": arguments["customer_id"],
                    "subject": arguments["subject"],
                    "priority": arguments.get("priority", "medium")}
        return {"error": f"Unknown tool: {name}"}

# Simulate the MCP interaction
server = SimulatedCRMMCPServer()

print("=== Simulated Custom MCP Server Interaction ===")
print()

# Step 1: Agent discovers tools
tools = server.list_tools()
print("1. Agent discovers tools:")
for tool in tools:
    print(f"   - {tool['name']}: {tool['description']}")

# Step 2: Agent calls lookup_customer
print()
print("2. Agent calls lookup_customer('C001'):")
result = server.call_tool("lookup_customer", {"customer_id": "C001"})
print(f"   Result: {json.dumps(result)}")

# Step 3: Agent calls create_ticket
print()
print("3. Agent calls create_ticket:")
result = server.call_tool("create_ticket", {
    "customer_id": "C001",
    "subject": "Order delayed by 2 weeks",
    "priority": "high"
})
print(f"   Result: {json.dumps(result)}")

### Expected Output

```
=== Simulated Custom MCP Server Interaction ===

1. Agent discovers tools:
   - lookup_customer: Look up customer by ID
   - create_ticket: Create a support ticket

2. Agent calls lookup_customer('C001'):
   Result: {"name": "Alice Johnson", "tier": "Gold", "email": "alice@example.com"}

3. Agent calls create_ticket:
   Result: {"ticket_id": "TKT-1001", "customer": "C001", "subject": "Order delayed by 2 weeks", "priority": "high"}
```

### What Just Happened

We simulated the **full MCP lifecycle**: discovery, understanding, and invocation. This is exactly what happens in production:
1. Agent connects to MCP server and asks "What tools do you have?"
2. Agent reads tool descriptions to understand capabilities
3. Agent calls the appropriate tool with the right parameters
4. Agent receives the result and incorporates it into its response

---

## Step 5: Configuring an Agent to Use MCP Tools

In the Databricks Agent Framework, you configure MCP servers in the agent's configuration. The agent automatically discovers tools from all connected MCP servers.

In [ ]:
# ---- Step 5: Agent configuration with MCP servers ----
# This shows how an agent's configuration references MCP servers.

agent_config = {
    "agent_name": "customer-support-agent",
    "llm_endpoint": "databricks-meta-llama-3-3-70b-instruct",
    "system_prompt": "You are a helpful customer support agent. Use the available tools to look up information and take actions.",
    "mcp_servers": [
        {
            "type": "managed",
            "name": "unity-catalog-mcp",
            "config": {
                "catalog": "main",
                "schema": "genai_labs",
                "description": "Provides order lookup and pricing tools via UC functions"
            }
        },
        {
            "type": "external",
            "name": "slack-mcp",
            "config": {
                "credentials": "{{secrets/support/slack-token}}",
                "allowed_channels": ["#support", "#escalations"],
                "description": "Send messages to Slack support channels"
            }
        },
        {
            "type": "custom",
            "name": "internal-crm-mcp",
            "config": {
                "endpoint_url": "https://crm-mcp.internal.company.com",
                "credentials": "{{secrets/support/crm-api-key}}",
                "description": "Access internal CRM for customer data and ticket creation"
            }
        }
    ]
}

print("=== Agent Configuration with MCP Servers ===")
print()
print(f"Agent: {agent_config['agent_name']}")
print(f"LLM:   {agent_config['llm_endpoint']}")
print()
print("Connected MCP Servers:")
for mcp in agent_config['mcp_servers']:
    print(f"  [{mcp['type'].upper():<8}] {mcp['name']}")
    print(f"             {mcp['config']['description']}")
print()
print("At runtime, the agent discovers all tools from all three servers")
print("and can use ANY of them to answer user questions.")

### Expected Output

```
=== Agent Configuration with MCP Servers ===

Agent: customer-support-agent
LLM:   databricks-meta-llama-3-3-70b-instruct

Connected MCP Servers:
  [MANAGED ] unity-catalog-mcp
             Provides order lookup and pricing tools via UC functions
  [EXTERNAL] slack-mcp
             Send messages to Slack support channels
  [CUSTOM  ] internal-crm-mcp
             Access internal CRM for customer data and ticket creation

At runtime, the agent discovers all tools from all three servers
and can use ANY of them to answer user questions.
```

### What Just Happened

We showed how an agent's configuration connects to **all three types** of MCP servers simultaneously. The agent merges the tool catalogs from all servers and can use any tool from any server. This is the power of MCP — **mix and match** managed, external, and custom tools.

---

## Step 6: Decision Framework — Managed vs External vs Custom

Choosing the right MCP server type is an exam-critical skill. Use this decision tree.

In [ ]:
# ---- Step 6: Decision framework with practice scenarios ----

decision_tree = """
Need to give an agent access to a tool?
    |
    +---> Is it a Databricks-native resource?
    |     (UC tables, functions, Genie SQL)
    |         YES --> Managed MCP (zero setup)
    |
    +---> Is it a common SaaS tool?
    |     (Slack, GitHub, Jira, Google Drive)
    |         YES --> External MCP (pre-built connector)
    |
    +---> Is it a proprietary/internal system?
    |     (Internal API, legacy DB, custom ML model)
              YES --> Custom MCP (build your own)
"""

print("=== MCP Server Decision Tree ===")
print(decision_tree)

# Practice scenarios
scenarios = [
    {
        "scenario": "Agent needs to query a Delta table for customer data",
        "answer": "Managed MCP (Unity Catalog)",
        "why": "Delta tables are Databricks-native -- UC MCP provides access automatically"
    },
    {
        "scenario": "Agent needs to post updates to a Slack channel",
        "answer": "External MCP (Slack)",
        "why": "Slack is a common SaaS tool with a pre-built MCP connector"
    },
    {
        "scenario": "Agent needs to call a company's proprietary pricing engine API",
        "answer": "Custom MCP",
        "why": "Proprietary API with no pre-built connector -- must build custom"
    },
    {
        "scenario": "Agent needs to run ad-hoc SQL queries against warehouse data",
        "answer": "Managed MCP (Genie)",
        "why": "Genie MCP provides NL-to-SQL on Databricks-native data"
    },
    {
        "scenario": "Agent needs to create GitHub issues for bug reports",
        "answer": "External MCP (GitHub)",
        "why": "GitHub is a common SaaS tool with a pre-built MCP connector"
    },
    {
        "scenario": "Agent needs to access a legacy on-premise ERP system",
        "answer": "Custom MCP",
        "why": "Legacy on-premise system with no standard connector -- must build custom"
    },
]

print("=== Practice: Choose the Right MCP Server Type ===")
print()
for i, s in enumerate(scenarios, 1):
    print(f"Scenario {i}: {s['scenario']}")
    print(f"  --> {s['answer']}")
    print(f"      {s['why']}")
    print()

### Expected Output

```
=== MCP Server Decision Tree ===

Need to give an agent access to a tool?
    |
    +---> Is it a Databricks-native resource?
    |         YES --> Managed MCP (zero setup)
    +---> Is it a common SaaS tool?
    |         YES --> External MCP (pre-built connector)
    +---> Is it a proprietary/internal system?
              YES --> Custom MCP (build your own)

=== Practice: Choose the Right MCP Server Type ===

Scenario 1: Agent needs to query a Delta table for customer data
  --> Managed MCP (Unity Catalog)
      Delta tables are Databricks-native -- UC MCP provides access automatically
...
```

### What Just Happened

We practiced the **decision framework** for choosing MCP server types. This is a direct exam skill — given a scenario, identify which MCP type fits. The rule is simple:
- **Databricks resource** --> Managed MCP
- **Common SaaS tool** --> External MCP
- **Proprietary/internal system** --> Custom MCP

---

## Exam Tips and Traps

| # | Tip or Trap | Explanation |
|---|-------------|-------------|
| 1 | **Trap:** "MCP is only for custom integrations" | Managed MCP servers for UC and Genie are built-in. Zero setup. |
| 2 | **Tip:** UC function COMMENT = MCP tool description | The LLM reads the COMMENT to decide when to use the tool. Write clear, actionable comments. |
| 3 | **Trap:** "Agents need hardcoded tool lists" | MCP tools are discovered dynamically at runtime. Add a new UC function, and agents can use it immediately. |
| 4 | **Tip:** MCP credentials use Databricks Secrets | Never put API keys in code. Use `{{secrets/scope/key}}` syntax. |
| 5 | **Trap:** "Custom MCP requires writing protocol code" | Use the MCP SDK. You just define tool functions — the SDK handles the protocol. |
| 6 | **Tip:** An agent can connect to MULTIPLE MCP servers | Mix managed + external + custom. The agent merges all tool catalogs. |

---

## Key Takeaways

### Core Concepts

- **MCP** is an open standard protocol for connecting agents to tools and data sources
- **Three types**: Managed (built-in), External (pre-built connectors), Custom (build your own)
- **Managed MCP**: Unity Catalog MCP (UC functions as tools), Genie MCP (NL-to-SQL)
- **External MCP**: Pre-built connectors for Slack, GitHub, Jira, etc.
- **Custom MCP**: You build a server using the MCP SDK for proprietary systems
- Agents **discover** tools at runtime — no hardcoding required

### Architecture Summary

```
Agent (MCP Client)
    |
    +---> Managed MCP (Unity Catalog) --> UC Functions, Tables
    |
    +---> Managed MCP (Genie) --> Natural Language SQL
    |
    +---> External MCP (Slack) --> Send Messages, Search
    |
    +---> Custom MCP (Internal CRM) --> Lookup, Create Ticket
```

### Decision Rule

```
Databricks resource?  -->  Managed MCP
Common SaaS tool?     -->  External MCP
Proprietary system?   -->  Custom MCP
```

---

## Cleanup (Optional)

Uncomment and run the following cell to remove the UC functions created in this lab.

In [ ]:
# ---- Cleanup: Remove lab functions (uncomment to run) ----
# spark.sql(f"DROP FUNCTION IF EXISTS {CATALOG}.{SCHEMA}.get_order_status")
# spark.sql(f"DROP FUNCTION IF EXISTS {CATALOG}.{SCHEMA}.calculate_discount")
# print("Cleanup complete.")

---

## Next Lab

Continue to **Lab 04-08: Apps and Interfaces** (`08_apps_and_interfaces.ipynb`), where you will learn about Databricks Apps, Playground, and serving endpoints — the three ways to deliver your GenAI application to users.